In [11]:
from discovery_utils.getters import openalex
from discovery_utils import PROJECT_DIR, logging
import pandas as pd
from itertools import combinations, chain
from itertools import product

In [17]:
domain_keywords = [
    "child", "children", "preschool", "infant", "early childhood", 
    "toddler", "under 5", "caregiver", "parent", "family", "baby",
]

measurement_keywords = [
    "tool", "instrument", "measure", "assessment", 
    "scale", "questionnaire", "survey", "screening", "observation"
]

def build_combined_query(domain_keywords, measure_keywords):
    """
    Generates a single boolean query string in the form of:
    (domain1 AND measure1) OR (domain1 AND measure2) OR ...
    """
    combinations = product(domain_keywords, measure_keywords)
    clauses = [f"({d} AND {m})" for d, m in combinations]
    combined_query = " OR ".join(clauses)
    return combined_query

# Generate the query
query_string = build_combined_query(domain_keywords, measurement_keywords)

# Print or use the query string
# print(query_string)

In [20]:
queries = [
    query_string
]

data_dfs = []
for query in queries:
    logging.info(f"Querying OpenAlex for '{query}'")
    data_df = (
        openalex.get_openalex_works(query, n_works = 10000)
        .assign(query=query)
    )
    logging.info(f"Collected {len(data_df)} works")
    data_dfs.append(data_df)
    
data_df = pd.concat(data_dfs, ignore_index=True)

2025-04-23 17:55:13,019 - root - INFO - Querying OpenAlex for '(child AND tool) OR (child AND instrument) OR (child AND measure) OR (child AND assessment) OR (child AND scale) OR (child AND questionnaire) OR (child AND survey) OR (child AND screening) OR (child AND observation) OR (children AND tool) OR (children AND instrument) OR (children AND measure) OR (children AND assessment) OR (children AND scale) OR (children AND questionnaire) OR (children AND survey) OR (children AND screening) OR (children AND observation) OR (preschool AND tool) OR (preschool AND instrument) OR (preschool AND measure) OR (preschool AND assessment) OR (preschool AND scale) OR (preschool AND questionnaire) OR (preschool AND survey) OR (preschool AND screening) OR (preschool AND observation) OR (infant AND tool) OR (infant AND instrument) OR (infant AND measure) OR (infant AND assessment) OR (infant AND scale) OR (infant AND questionnaire) OR (infant AND survey) OR (infant AND screening) OR (infant AND obser

In [21]:
_data_df = data_df.drop_duplicates("id")
len(_data_df)

9545

In [23]:
data_df.to_csv("afs_open_alex_scan.csv", index=False)